In [ ]:
# Cell 1: Import libraries and configure dependencies
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

import os
import numpy as np
import h5py, scipy.io as sio
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model
from pathlib import Path
import yaml

# ------------------------------
# Resolve notebook directory safely
# ------------------------------
notebook_dir = Path().resolve()
project_root = notebook_dir.parent

print(f"Notebook directory: {notebook_dir}")
print(f"Project root: {project_root}")

# ------------------------------
# Load DeepRadar2022 dataset
# ------------------------------
cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    path = Path(local_cfg.get('dataset_root', '/scratch/rameyjm7/datasets')) / 'DeepRadar2022'
else:
    path = Path('/scratch/rameyjm7/datasets/DeepRadar2022')
print(f"Loading DeepRadar2022 from: {path}")

def load_h5(filepath, key):
    with h5py.File(filepath, "r") as f:
        return np.array(f[key], dtype="float32").T

def load_mat(filepath, key):
    d = sio.loadmat(filepath)
    return d[key]

X_test  = load_h5(path / "X_test.mat", "X_test")
Y_test  = load_mat(path / "Y_test.mat", "Y_test")
lbl_test = load_mat(path / "lbl_test.mat", "lbl_test")

# Extract modulation class and SNR
cls_test, snr_test = lbl_test[:,0].astype(int)-1, lbl_test[:,1]

# ------------------------------
# Normalize IQ per sample
# ------------------------------
def normalize_iq(X):
    Xn = np.empty_like(X)
    for i in range(X.shape[0]):
        scale = np.max(np.abs(X[i])) + 1e-12
        Xn[i] = X[i] / scale
    return Xn

X_test = normalize_iq(X_test)

# ------------------------------
# Append SNR as a third channel
# ------------------------------
def append_snr_feature(X, snr):
    X_out = []
    for i in range(X.shape[0]):
        snr_col = np.full((X.shape[1], 1), snr[i] / 20.0)
        X_out.append(np.concatenate([X[i], snr_col], axis=1))
    return np.array(X_out, dtype=np.float32)

X_test = append_snr_feature(X_test, snr_test)

# ------------------------------
# Load pre-trained model
# ------------------------------
model_path = project_root / "models" / "deepradar2022" / "deepradar2022_cnn_bilstm_final.keras"
print(f"Loading model from: {model_path}")
model = load_model(model_path)

# ------------------------------
# Evaluate model
# ------------------------------
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print(f"\n✅ Model Test Accuracy (all SNRs): {acc*100:.2f}%")

# ------------------------------
# Predictions and metrics
# ------------------------------
Y_pred = model.predict(X_test, verbose=0)
y_true = np.argmax(Y_test, axis=1)
y_pred = np.argmax(Y_pred, axis=1)

# ------------------------------
# Confusion Matrix
# ------------------------------
labels = [
    "LFM", "2FSK", "4FSK", "8FSK", "Costas", "2PSK", "4PSK", "8PSK",
    "Barker", "Huffman", "Frank", "P1", "P2", "P3", "P4", "Px",
    "Zadoff-Chu", "T1", "T2", "T3", "T4", "NM", "Noise"
]

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap="Blues", cbar=True,
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (CNN + BiLSTM, DeepRadar2022)")
plt.tight_layout()
plt.show()

print("\nClassification Report (All SNRs):")
print(classification_report(y_true, y_pred, target_names=labels))

# ------------------------------
# Accuracy vs SNR
# ------------------------------
unique_snrs = sorted(np.unique(snr_test))
acc_snr = []
for snr in unique_snrs:
    idx = np.where(snr_test == snr)[0]
    acc_snr.append(accuracy_score(y_true[idx], y_pred[idx]) * 100)

plt.figure(figsize=(8,5))
plt.plot(unique_snrs, acc_snr, 'b-o')
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy (%)")
plt.title("Recognition Accuracy vs SNR (DeepRadar2022 CNN+BiLSTM)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 2: Import libraries and configure dependencies
# https://www.kaggle.com/jacobramey
# https://github.com/rameyjm7

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model
import tensorflow as tf
import os
import yaml

# --------------------------------------------------------------
# Resolve model path
# --------------------------------------------------------------
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_path = os.path.join(project_root, "models", "rml2016", "rml2016_lstm_rnn_2024.keras")

print("Resolved model path:", model_path)
assert os.path.exists(model_path), f"Model file not found: {model_path}"

model = load_model(model_path)
print("Model loaded successfully.")

# --------------------------------------------------------------
# Load RML2016.10a Dataset
# --------------------------------------------------------------
cfg_path = Path(project_root) / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    local_cfg = yaml.safe_load(cfg_path.read_text())
    pkl_path = local_cfg.get('datasets', {}).get('rml2016', {}).get('pkl', '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl')
else:
    pkl_path = '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl'
print("Loading dataset:", pkl_path)

with open(pkl_path, "rb") as f:
    data = pickle.load(f, encoding="latin1")

# --------------------------------------------------------------
# Prepare Data (your exact provided format)
# --------------------------------------------------------------
def prepare_data(data):
    X, y, snrs = [], [], []

    for (mod_type, snr), signals in data.items():
        for signal in signals:
            # IQ: shape (128, 2)
            iq = np.vstack([signal[0], signal[1]]).T

            # SNR feature channel (raw SNR, consistent with your training format)
            snr_col = np.full((128, 1), snr, dtype=np.float32)

            combined = np.hstack([iq, snr_col])  # (128, 3)

            X.append(combined)
            y.append(mod_type)
            snrs.append(snr)   # keep real SNR for analysis

    X = np.array(X)
    y = np.array(y)
    snrs = np.array(snrs)

    # Encode labels
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    # Train/test split
    X_train, X_test, y_train, y_test, snr_train, snr_test = train_test_split(
        X, y_encoded, snrs, test_size=0.2, random_state=42
    )

    # LSTM requires this shape already (128, 3)
    return X_train, X_test, y_train, y_test, snr_train, snr_test, encoder

# Prepare
X_train, X_test, y_train, y_test, snr_train, snr_test, encoder = prepare_data(data)

# --------------------------------------------------------------
# Evaluate model on full test set
# --------------------------------------------------------------
y_pred = np.argmax(model.predict(X_test, verbose=False), axis=1)

# Confusion matrix (ALL SNR)
cm_all = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_all, annot=True, fmt="d", cmap="Blues",
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (All SNR Levels)")
plt.show()

print("\nClassification Report (All SNR Levels):")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

# --------------------------------------------------------------
# Evaluate only SNR > 5 dB subset
# --------------------------------------------------------------
idx_high = np.where(snr_test > 5)[0]

X_high = X_test[idx_high]
y_high = y_test[idx_high]

print(f"\nSamples with SNR > 5 dB: {len(idx_high)}")

y_pred_high = np.argmax(model.predict(X_high, verbose=False), axis=1)

cm_high = confusion_matrix(y_high, y_pred_high)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_high, annot=True, fmt="d", cmap="Blues",
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (SNR > 5 dB)")
plt.show()

print("\nClassification Report (SNR > 5 dB):")
print(classification_report(y_high, y_pred_high, target_names=encoder.classes_))
